# TensorFlow Tutorial Hands-On: Regression
This notebook is aimed to put [the TensorFlow tutorial of a regression problem](https://www.tensorflow.org/tutorials/keras/regression) into practice with the dataset created in [dataset.ipynb](./dataset.ipynb).

In [ ]:
import random

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.subplots as ps
import tensorflow as tf

pio.renderers.default = "svg"

__version__ = "0.0.0"

## Pre-Processing
First load a dataset using pandas, and split it into training and test sets.

Normilize features if necessary.

In [ ]:
repo = "https://github.com/wasedatakeuchilab/DL-SCDC"

# Load the dataset
dataset_name = "dataset"
dataset_version = "0.0.0"
df = pd.read_csv(
    f"{repo}/raw/{dataset_name}/v{dataset_version}/datasets/{dataset_name}.csv.gz"
)
dataset = list(df.assign(label=list(zip(df["t0"], df["y0"]))).groupby("id"))

# Split the dataset into training and test sets
random.seed(0)
random.shuffle(dataset)
n = int(0.8 * len(dataset))
train_dataset = dataset[:n]
test_dataset = dataset[n:]


def split_dataset(dataset):
    label = np.array([list(df.label)[0] for _, df in dataset])
    data = np.array([list(df.y) for _, df in dataset])
    return label, data


# Seperate label from features
train_label, train_data = split_dataset(train_dataset)
test_label, test_data = split_dataset(test_dataset)
print("Size of dataset:", len(dataset))

## Model
Build a model with the following hyperparameters.

- 4 layers
    - 200 units, activation=tanh
    - 120 units, activation=tanh
    - 30 units, activation=tanh
    - 2 units, activation=sigmoid
- MSE(Mean Squared Error) for optimization
- Adam optimizer

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(len(train_data[0])),
        tf.keras.layers.Dense(200, activation="tanh"),
        tf.keras.layers.Dense(120, activation="tanh"),
        tf.keras.layers.Dense(30, activation="tanh"),
        tf.keras.layers.Dense(2, activation="sigmoid"),
    ]
)
model.compile(
    loss="mse",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["mse", "mae"],
)
model.summary()

## Training
Train the model with the training dataset.

20% of the dataset is used for validation.

Early stopping is used for proper number of epochs.

In [ ]:
def schedule_learning_rate(epoch: int) -> float:
    if epoch <= 50:
        return 1e-4
    else:
        return 1e-5


history = model.fit(
    train_data,
    train_label,
    epochs=1000,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20),
        tf.keras.callbacks.LearningRateScheduler(schedule_learning_rate),
    ],
    verbose=0,
)

## Evaluate
Evaluate the trained model.

In [ ]:
loss, mse, mae = model.evaluate(test_data, test_label)

## Visualize
Plot the model's training process using Plotly.

In [ ]:
hist = pd.DataFrame(history.history).assign(epoch=history.epoch)
ps.make_subplots(rows=1, cols=2).add_scatter(
    x=hist["epoch"], y=hist["mse"], name="mse", row=1, col=1
).add_scatter(
    x=hist["epoch"], y=hist["val_mse"], name="val_mse", row=1, col=1
).update_yaxes(
    title="Mean Squared Error", type="log", row=1, col=1
).add_scatter(
    x=hist["epoch"], y=hist["mae"], name="mae", row=1, col=2
).add_scatter(
    x=hist["epoch"], y=hist["val_mae"], name="val_mae", row=1, col=2
).update_yaxes(
    title="Mean Absolute Error", type="log", row=1, col=2
).update_xaxes(
    title="Epoch"
).show(
    width=1000
)

Plot predected values vs. true values.

In [ ]:
predictions = model.predict(test_data, verbose=0)
ps.make_subplots(rows=1, cols=2).add_scatter(
    x=test_label[:, 0], y=predictions[:, 0], mode="markers", name="t0", row=1, col=1
).update_xaxes(title="True t0", range=(0, 0.3), row=1, col=1).update_yaxes(
    title="Predicted t0", range=(0, 0.3), row=1, col=1
).add_scatter(
    x=test_label[:, 1], y=predictions[:, 1], mode="markers", name="y0", row=1, col=2
).update_xaxes(
    title="True y0", range=(0, 1.0), row=1, col=2
).update_yaxes(
    title="Predicted y0", range=(0, 1.0), row=1, col=2
).add_scatter(
    x=[0, 1], y=[0, 1], line=dict(color="black"), name="y=x", row="all", col="all"
).update_layout(
    showlegend=False
).show(
    width=1000
)

See results as plots.

In [ ]:
for data in random.sample(list(test_data), 5):
    prediction, *_ = model.predict(np.expand_dims(data, 0), verbose=0)
    d = (1.0 - prediction[1]) * 0.1
    px.line(y=data, range_y=(prediction[1] - d, 1 + d)).add_vline(
        prediction[0] * len(data)
    ).add_hline(prediction[1]).show()

# Save
Save the trained model.

In [ ]:
filename = f"tutorial"
model.save(f"models/{filename}")
!tar czf models/"$filename".tar.gz -C models $filename

Copyright (c) 2022 Shuhei Nitta